# Exploratory Data Analysis (EDA) - Players 
En este documento analizamos el dataset condensado por jugador (`players.csv`). A diferencia del historial, este conjunto contiene los acumulados y promedios por temporada de cada futbolista. Realizaremos limpieza estructurada y usaremos 3 dimensiones para estudiar el verdadero valor, inteligencia táctica y rendimiento 'Per 90' de cada perfil.

In [ ]:
import os
# --- SOLUCIÓN DE RUTAS (CWD) 100% PORTABLE ---
if os.path.exists('CSV (API)') and os.path.exists('EDA (Análisis Exploratorio de Datos)'):
    os.chdir('EDA (Análisis Exploratorio de Datos)')
print('📁 Directorio configurado en:', os.getcwd())


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

## 1. Carga y Limpieza de Datos 
- **Clasificación Posicional**: Mapeamos los IDs numéricos de posiciones a nombres legibles (Arquero, Defensa, Mediocampista, Delantero).
- **Valores Nulos**: En FPL, los jugadores 100% sanos tienen `null` en "Probabilidad de jugar la próxima ronda". Imputamos esos vacíos inteligentemente con `100`.
- **Filtro de Ruidos**: Descartamos a los jugadores sin minutos para limpiar las visualizaciones de "jugadores fantasma".


In [ ]:
df = pd.read_csv('../CSV (API)/players.csv')

# 1. Imputación de Valores Nulos (Salud y Estado)
df['chance_of_playing_next_round'] = pd.to_numeric(df['chance_of_playing_next_round'], errors='coerce').fillna(100.0)
df['news'] = df['news'].fillna('Fit')

# 2. Arreglo de Tipos de Datos "Per 90" (A veces vienen como string en FPL)
per90_cols = ['xG_per90', 'xA_per90', 'points_per_game', 'selected_by_percent']
for col in per90_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# Mapeo de Posiciones FPL
# 1: Portero, 2: Defensa, 3: Medio, 4: Delantero
pos_map = {1: 'Portero', 2: 'Defensa', 3: 'Mediocampista', 4: 'Delantero'}
if 'position' in df.columns and df['position'].dtype in [np.int64, np.float64, int, float]:
    df['position_name'] = df['position'].map(pos_map)
else:
    df['position_name'] = df['position'] # En caso de que ya sea string

# Quedarnos solo con jugadores activos para no contaminar las métricas de rendimiento
df_active = df[df['minutes'] > 90].copy()

print(f"Dataset cargado y limpio. Jugadores Activos analizados: {df_active.shape[0]}\n")
df_active.head(3)

## Exploración Visual 1: Presupuesto vs Utilidad Absoluta 
**Objetivo**: Observar la eficiencia del costo de los jugadores. Comparamos el Precio (X), contra su Resistencia Física (Minutos, Y), y su Retorno Inversor (Puntos, Z).

**Insights:**
- Al colorear por posición, podemos rotar la cámara y observar la base del gráfico. Los **Porteros** y **Defensas** dominan el rango de precios bajos y minutos súper altos, generando una enorme columna vertebral de Puntos Z sólida a precios de subasta.
- Los **Delanteros** (Premium) crean una nube flotante extrema a la derecha (Alto Precio). Algunos justifican su precio alcanzando la atmósfera en Puntos Totales, pero muchos otros se hunden a mitad de tabla, consumiendo enormes salarios sin retorno por ser propensos a lesiones (Minutos Y bajos).


In [ ]:
fig1 = px.scatter_3d(
    df_active, 
    x='price', y='minutes', z='total_points',
    color='position_name',
    labels={'price': 'Precio Venta (X)', 'minutes': 'Minutos Jugados (Y)', 'total_points': 'Puntos FPL Totales (Z)', 'position_name': 'Posición'},
    hover_name='web_name',
    hover_data=['team_short', 'goals_scored', 'assists'],
    opacity=0.8,
    title='Rendimiento de Mercado: Puntos vs Precio y Resistencia'
)
fig1.update_traces(marker=dict(size=5, line=dict(width=0.5, color='gray')))
fig1.update_layout(margin=dict(l=0, r=0, b=0, t=40), scene=dict(camera=dict(eye=dict(x=-1.5, y=1.5, z=0.5))))
fig1.show()

## Exploración Visual 2: Análisis Estructural (ICT Index) 
**Objetivo**: El "ICT Index" combina Influencia (pases clave, intercepciones), Creatividad (creación de gol), y Amenaza (Oportunidades de tiro). Evaluamos cómo estos tres pilares se cruzan para fabricar un monstruo en Fantasy.

**Insights:**
- Es espectacular cómo el volumen 3D desgrana a los jugadores: En el eje de creatividad (Y) dominan los extremos/mediapuntas (creadores). En el eje de Amenaza (Z) escalan los nueves puros. 
- Los "Jugadores Franquicia" (que deslumbran con Puntos altísimos en color brillante) son aquellos raros especímenes matemáticos que logran desprenderse de los ejes y viajar por la diagonal perfecta hacia el centro del cubo, acumulando altos niveles en las tres métricas maestras simultáneamente.


In [ ]:
fig2 = px.scatter_3d(
    df_active,
    x='influence', y='creativity', z='threat',
    color='total_points',
    color_continuous_scale='Magma',
    labels={'influence': 'Influencia (X)', 'creativity': 'Creatividad (Y)', 'threat': 'Amenaza Ofensiva (Z)', 'total_points': 'Puntos Obtenidos'},
    hover_name='web_name',
    hover_data=['position_name', 'ict_index'],
    title='Radiografía del Rendimiento: Influencia, Creatividad y Amenaza'
)
fig2.update_traces(marker=dict(size=5, opacity=0.9))
fig2.update_layout(margin=dict(l=0, r=0, b=0, t=40), scene=dict(camera=dict(eye=dict(x=1.5, y=-1.5, z=0.5))))
fig2.show()

## Exploración Visual 3: Eficiencia Táctica "Per 90" (Puntos por Partido) 
**Objetivo**: Muchos jugadores lucen bien jugando 38 partidos seguidos. ¿Pero quiénes son verdaderamente letales por cada 90 minutos netos que pisan el césped? Utilizamos el tamaño de las esferas para representar si jugaron toda la temporada (Esferas grandes) o solo unos partidos (Esferitas).

**Insights:**
- Se exponen maravillosamente los "Espejismos". Hay puntitos minúsculos que vuelan alto en xG per 90 (X) y Puntos por Partido (Z), representando suplentes o recambios que anotaron un gol en 20 minutos jugados e inflaron su promedio artificalmente.
- Los reyes del campeonato son las **Esferas Gigantes flotantes**: Jugadores con altísima participación, pero que mantienen un xG_per90 elevadísimo a lo largo de 30+ partidos, soportando el cansancio y manteniendo la cuota goleadora de forma robótica.


In [ ]:
fig3 = px.scatter_3d(
    df_active,
    x='xG_per90', y='xA_per90', z='points_per_game',
    color='position_name',
    size='minutes', size_max=20,
    labels={'xG_per90': 'Goles Esperados c/90m (X)', 'xA_per90': 'Asistencias Esperadas c/90m (Y)', 'points_per_game': 'Puntos por Partido (Z)', 'position_name': 'Posición'},
    hover_name='web_name',
    hover_data=['team_short', 'minutes', 'goals_scored'],
    title='Letalidad Proporcional: Rendimiento Matemático cada 90 Minutos'
)
fig3.update_traces(marker=dict(opacity=0.7, line=dict(width=0.2, color='white')))
fig3.update_layout(margin=dict(l=0, r=0, b=0, t=40), scene=dict(camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.5))))
fig3.show()